<a href="https://colab.research.google.com/github/Ginkno/postech-tech-challenge-fase2-grupo1-12dtat/blob/Elizabeth/notebooks/03_modelagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [16]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW_1 = Path("/content") / "data" / "raw" / "application_record.csv"
RAW_2 = Path("/content") / "data" / "raw" / "credit_record.csv"

PROCESSED = Path("/content") / "data" / "processed" / "dataset_tratado.csv"
TARGET = "IS_BAD_PAYER"

pd.set_option("display.max_columns", None)

In [17]:
df_model = pd.read_csv(PROCESSED)

In [18]:
display(df_model.head())

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED,TOTAL_MOUNTHS_OBSERVED,STATUS_LATED,STATUS_ON_TIME,STATUS_NO_CREDIT,WORST_LATE_HISTORY,IS_BAD_PAYER,INCOME_PER_PERSON,HAS_CHILDREN,PROP_LATED,PROP_ON_TIME,PROP_NO_CREDIT,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff
0,1,1,1,0,427500.0,1,1,0,0,2.0,32,0,12,16,2,13,1,2,1,213750.0,0,0.125000,0.812500,0.062500,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,1,1,1,0,427500.0,1,1,0,0,2.0,32,0,12,15,2,12,1,2,1,213750.0,0,0.133333,0.800000,0.066667,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,1,1,1,0,112500.0,1,0,0,0,2.0,58,0,3,30,7,7,16,1,0,56250.0,0,0.233333,0.233333,0.533333,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
3,0,0,1,0,270000.0,1,0,1,1,1.0,52,0,8,5,2,0,3,1,0,270000.0,0,0.400000,0.000000,0.600000,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
4,0,0,1,0,270000.0,1,0,1,1,1.0,52,0,8,5,0,0,5,0,0,270000.0,0,0.000000,0.000000,1.000000,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [19]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['IS_BAD_PAYER', 'WORST_LATE_HISTORY'])
y = df_model['IS_BAD_PAYER']

#Dividindo os dados em 80% para treino e 20% para teste.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Shape de X_train: {X_train.shape}')
print(f'Shape de X_test: {X_test.shape}')
print(f'Shape de y_train: {y_train.shape}')
print(f'Shape de y_test: {y_test.shape}')

print('\nDistribuição da variável alvo no treino:')
display(y_train.value_counts(normalize=True))

print('\nDistribuição da variável alvo no teste:')
display(y_test.value_counts(normalize=True))

Shape de X_train: (29165, 57)
Shape de X_test: (7292, 57)
Shape de y_train: (29165,)
Shape de y_test: (7292,)

Distribuição da variável alvo no treino:


,proportion
IS_BAD_PAYER,
0,0.88229
1,0.11771



Distribuição da variável alvo no teste:


,proportion
IS_BAD_PAYER,
0,0.882337
1,0.117663


## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [20]:
import numpy as np

# Calcular scale_pos_weight para lidar com o desbalanceamento de classes
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight_value = neg_count / pos_count

print(f"Scale Pos Weight calculado: {scale_pos_weight_value:.2f}")

Scale Pos Weight calculado: 7.50


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer

# Novos modelos para adicionar
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import numpy as np

# --- Definindo as transformações de pré-processamento ---

# Transformação logarítmica para 'AMT_INCOME_TOTAL'
def log_transform(X):
    # Adiciona um pequeno valor para evitar log(0), se necessário,
    # mas np.log1p já trata isso
    return np.log1p(X)

log_transformer = FunctionTransformer(log_transform, validate=False)

# Capping para 'CNT_CHILDREN' e 'CNT_FAM_MEMBERS'
# Os valores de capping foram ajustados com base na análise
# de outliers no Notebook 01.
upper_bound_children = 2.50
upper_bound_family = 4.50

def cap_children(X):
    return np.clip(X, a_min=None, a_max=upper_bound_children)

def cap_family_members(X):
    return np.clip(X, a_min=None, a_max=upper_bound_family)

cap_children_transformer = FunctionTransformer(cap_children, validate=False)
cap_family_members_transformer = FunctionTransformer(cap_family_members, validate=False)

# Criando o ColumnTransformer para aplicar as transformações.
# 'remainder='passthrough'' garante que as outras colunas não sejam afetadas.
preprocessor = ColumnTransformer(
    transformers=[
        ('log_income', log_transformer, ['AMT_INCOME_TOTAL']),
        ('cap_children', cap_children_transformer, ['CNT_CHILDREN']),
        ('cap_family_members', cap_family_members_transformer, ['CNT_FAM_MEMBERS'])
    ],
    remainder='passthrough' # Mantenha as colunas não transformadas
)

# Definindo o passo de balanceamento (mantido para modelos que o usam)
# class_weight = 'balanced' (removido para modelos que não usam)

modelos = {
    "logistic_regression": Pipeline([
        ("preprocessor", preprocessor),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight='balanced',random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("preprocessor", preprocessor), # RF pode se beneficiar de features transformadas
        ("scaler", StandardScaler()), # É uma boa prática escalar mesmo para RF após transformações
        ("clf", RandomForestClassifier(n_estimators=300, class_weight='balanced',random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "svm": Pipeline([
        ("preprocessor", preprocessor),
        ("scaler", StandardScaler()),
        ("clf", SVC(class_weight='balanced',random_state=RANDOM_STATE)),
    ]),
    "xgboost": Pipeline([
        ("preprocessor", preprocessor),
        ("scaler", StandardScaler()), # XGBoost pode se beneficiar de features escaladas, embora não seja estritamente necessário para árvores.
        ("clf", XGBClassifier(scale_pos_weight=scale_pos_weight_value,random_state=RANDOM_STATE, eval_metric='logloss')),
    ]),
    "lightgbm": Pipeline([
        ("preprocessor", preprocessor),
        ("scaler", StandardScaler()), # LightGBM também pode se beneficiar de features escaladas.
        ("clf", LGBMClassifier(scale_pos_weight=scale_pos_weight_value,random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

## 3. Validação cruzada

In [23]:
from sklearn.model_selection import cross_val_score
import pandas as pd
from scipy import stats

all_model_scores = {}

for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    all_model_scores[nome] = scores # Armazena os scores de cada fold
    print(f"{nome}: {scores.mean().round(4)}")

logistic_regression: 0.322
random_forest: 0.3492
svm: 0.3626
xgboost: 0.3967
[LightGBM] [Info] Number of positive: 2747, number of negative: 20585
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007535 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1635
[LightGBM] [Info] Number of data points in the train set: 23332, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.117735 -> initscore=-2.014053
[LightGBM] [Info] Start training from score -2.014053


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2747, number of negative: 20585
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007690 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1635
[LightGBM] [Info] Number of data points in the train set: 23332, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.117735 -> initscore=-2.014053
[LightGBM] [Info] Start training from score -2.014053


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2746, number of negative: 20586
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007611 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1640
[LightGBM] [Info] Number of data points in the train set: 23332, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.117692 -> initscore=-2.014466
[LightGBM] [Info] Start training from score -2.014466


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2746, number of negative: 20586
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007478 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1640
[LightGBM] [Info] Number of data points in the train set: 23332, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.117692 -> initscore=-2.014466
[LightGBM] [Info] Start training from score -2.014466


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2746, number of negative: 20586
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007619 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1637
[LightGBM] [Info] Number of data points in the train set: 23332, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.117692 -> initscore=-2.014466
[LightGBM] [Info] Start training from score -2.014466
lightgbm: 0.3804


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 4. Comparação

**Leitura:**  O F1-Score médio de todos os modelos ficou abaixo de 0,40. Como os algoritmos foram configurados com penalidades de peso (class_weight='balanced' / scale_pos_weight) para priorizar a identificação dos maus pagadores houve um aumento esperado de "falsos positivos", assim derrubou a precisão do modelo. O XGBoost venceu. A margem sobre o segundo colocado(LightGBM) foi de aproximadamente 0.0163 (0.3967 - 0.3804). A diferença entre o XGBoost e o LightGBM, aproximadamente 1.6 pontos percentuais, é relativamente pequena. Faremos os testes estatísticos  formais para afirmar se essa diferença é significante ou se está dentro do ruído de variação.

Teste T Pareado para Comparação de Modelos
Vamos realizar um teste t pareado entre os scores do XGBoost e do LightGBM para verificar se a diferença observada é estatisticamente significativa.

In [25]:
from scipy import stats

# Obter os scores dos dois modelos a serem comparados
scores_xgboost = all_model_scores['xgboost']
scores_lightgbm = all_model_scores['lightgbm']

# Realizar o teste t pareado
# O parâmetro 'alternative' define o tipo de teste (two-sided, less, greater)
# Aqui, 'two-sided' verifica se há uma diferença significativa em qualquer direção.
t_statistic, p_value = stats.ttest_rel(scores_xgboost, scores_lightgbm, alternative='greater')

print(f"F1-scores XGBoost: {scores_xgboost}")
print(f"F1-scores LightGBM: {scores_lightgbm}")
print(f"Estatística t: {t_statistic:.4f}")
print(f"Valor p: {p_value:.4f}")

# Interpretação do resultado
alpha = 0.05 # Nível de significância

if p_value < alpha:
    print(f"Com p-valor ({p_value:.4f}) menor que {alpha}, podemos rejeitar a hipótese nula.")
    print("Isso sugere que o XGBoost tem um F1-score significativamente maior que o LightGBM.")
else:
    print(f"Com p-valor ({p_value:.4f}) maior que {alpha}, não temos evidências suficientes para rejeitar a hipótese nula.")
    print("A diferença observada nos F1-scores entre XGBoost e LightGBM pode ser devido ao acaso (ruído).")

F1-scores XGBoost: [0.41179314 0.40075793 0.3946102  0.39553281 0.38067633]
F1-scores LightGBM: [0.3921869  0.3797856  0.37393986 0.38428956 0.37172979]
Estatística t: 6.3519
Valor p: 0.0016
Com p-valor (0.0016) menor que 0.05, podemos rejeitar a hipótese nula.
Isso sugere que o XGBoost tem um F1-score significativamente maior que o LightGBM.


Como 0.0016 (p-value) é menor que 0.05 (alpha), rejeitamos a hipótese nula. Isso nos permite concluir que o XGBoost realmente possui um F1-score significativamente maior que o LightGBM neste conjunto de dados e com esta configuração de validação cruzada. Portanto, a diferença de 1.6 pontos percentuais não é ruído; ela é estatisticamente significativa.